In [304]:
import pandas as pd
import numpy as np


In [305]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [306]:
PROJECT_PATH = "/content/drive/MyDrive/UK_Data_Analyst_Market"

In [307]:
df_raw = pd.read_csv(
    f"{PROJECT_PATH}/data/raw/adzuna_jobs_raw.csv"
)

In [308]:
df_raw.shape

(532, 17)

In [309]:
df_clean = df_raw.copy()

In [310]:
df_jobs = (
    df_clean
    .drop_duplicates(subset="job_id")
    .reset_index(drop=True)
    .copy()
)

df_jobs.shape

(491, 17)

In [311]:
job_search_terms = (
    df_clean[
        ["job_id", "search_term"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

job_search_terms.head()

,job_id,search_term
0,5840324082,data analyst
1,5840324089,data analyst
2,5840324100,data analyst
3,5835920078,data analyst
4,5829131501,data analyst


In [312]:
print("Raw rows:", len(df_raw))
print("Unique jobs:", len(df_jobs))
print("Job-search term relationships:", len(job_search_terms))

Raw rows: 532
Unique jobs: 491
Job-search term relationships: 532


In [313]:
df_jobs["created"] = pd.to_datetime(
    df_jobs["created"],
    utc=True,
    errors="coerce"
)

df_jobs["collection_date"] = pd.to_datetime(
    df_jobs["collection_date"],
    errors="coerce"
)

df_jobs["salary_is_predicted"] = pd.to_numeric(
    df_jobs["salary_is_predicted"],
    errors="coerce"
).astype("Int64")

In [314]:
df_jobs.dtypes

,0
job_id,int64
title,object
company,object
location,object
created,"datetime64[ns, UTC]"
description,object
salary_min,float64
salary_max,float64
salary_is_predicted,Int64
latitude,float64


In [315]:
low_salary_jobs = df_jobs[
    (df_jobs["salary_min"] < 10000) |
    (df_jobs["salary_max"] < 10000)
][
    [
        "job_id",
        "title",
        "company",
        "salary_min",
        "salary_max",
        "salary_is_predicted",
        "description"
    ]
].sort_values("salary_min")

low_salary_jobs

,job_id,title,company,salary_min,salary_max,salary_is_predicted,description
99,5829393076,Data Analyst,Red Recruitment,0.0,31200.0,0,"Data Analyst Red Recruitment is recruiting a switched-on and detail-oriented Data Analyst to join an innovative fintech organisation in Bristol on a four-week temporary contract. Working within the Product team, you will help ensure the data used by an AI-powered platform is reliable and accurate. You will write and refine instructions for AI models, test data extractions and manually review the results. This is an excellent opportunity for someone who is curious about AI, enjoys solving proble…"
122,5840293769,Data Analyst,TXP,0.0,124280.0,0,"Data Analyst Location: Dunstable (2 days per week onsite) Duration: 3 Month Contract Start Date: ASAP Rate: £478 per day Inside IR35 Overview We are seeking an experienced Data Analyst to join a business-critical Martech transformation project. The successful candidate will play a key role in analysing, reconciling, and consolidating large-scale customer data sets across multiple systems to support the migration into a new marketing technology environment. This is an exciting opportunity for a …"
387,5724380833,Senior Reporting and Insights Analyst,Clear Business,0.0,57000.0,0,"Are you driven by data and inspired by innovation? In this pivotal role, you'll harness both established and emerging technologies to deliver powerful reporting and actionable insights that shape strategic decisions across the business. Your work will not only drive performance, it will also elevate our internal data culture, build capability and confidence across teams. As a trusted expert, you&rsquo;ll: Lead the creation of dynamic dashboards, visualisations, and reports that bring clarity to…"
314,5826815795,Financial Reporting Analyst,Line Up Aviation,0.0,76980.0,0,"Our client has an opportunity for a Financial Analyst to join them on a contract basis for 6 months. You will be supporting on the financial reporting on defence projects and support to future bid campaigns. Role : Financial Analyst Location : Oxford, minimum 3 days on site per week, could be more depending on project needs. Hours : 37.5 hours per week, Monday to Friday 08:30-17:00 Clearance : Must be eligible for MOD Security Clearance Hourly Rate : £37.01 per hour via Umbrella, inside IR35 Wh…"
240,5827489859,BI BA – Business Analyst with Business Intelligence (IT),Nexus Jobs Limited,450.0,450.0,0,BI BA ? Business Analyst with Business Intelligence Our Client is a leader in the pharmaceutical world globally. They are looking to recruit a BI Business Analyst with at least 5 to 10 years proven expertise. This role will be working very closely with the finance division and therefore requires you to have very good understanding of the processes within the finance division particularly in manufacturing. You will be responsible for producing financial reports as a BI Business Analyst. Power BI…
183,5822552201,Data Analyst,Hays Technology,9100.0,130000.0,0,"Data AnalystLocation: Milton Keynes (Hybrid, minimum 2 days onsite) We're looking for an experienced Data Analyst to join a growing data team, supporting the development of modern data solutions, reporting capabilities, and analytics across a large-scale Azure data platform. The role combines data modelling, reporting, and hands-on data engineering work within a cloud-based environment. Key Responsibilities Design and develop data models, schemas, KPIs, measures, calculations, and critical data…"


In [316]:
df_jobs["salary_is_predicted"].value_counts(dropna=False)

,count
salary_is_predicted,
0,271
1,220


In [317]:
df_jobs = df_jobs.drop(columns=["search_term"])

In [318]:
print("Unique jobs:", df_jobs.shape)
print("Job-search relationships:", job_search_terms.shape)

Unique jobs: (491, 16)
Job-search relationships: (532, 2)


In [319]:
df_jobs["salary_source"] = (
    df_jobs["salary_is_predicted"]
    .map({
        0: "Advertised / non-predicted",
        1: "Adzuna predicted"
    })
)

In [320]:
df_jobs["salary_source"].value_counts()

,count
salary_source,
Advertised / non-predicted,271
Adzuna predicted,220


In [321]:
pd.set_option("display.max_colwidth", None)

In [322]:
for _, row in low_salary_jobs.iterrows():
    print("=" * 100)
    print("JOB ID:", row["job_id"])
    print("TITLE:", row["title"])
    print("COMPANY:", row["company"])
    print("SALARY:", row["salary_min"], "-", row["salary_max"])
    print("PREDICTED:", row["salary_is_predicted"])
    print()
    print(row["description"])
    print()

JOB ID: 5829393076
TITLE: Data Analyst
COMPANY: Red Recruitment
SALARY: 0.0 - 31200.0
PREDICTED: 0

Data Analyst Red Recruitment is recruiting a switched-on and detail-oriented Data Analyst to join an innovative fintech organisation in Bristol on a four-week temporary contract. Working within the Product team, you will help ensure the data used by an AI-powered platform is reliable and accurate. You will write and refine instructions for AI models, test data extractions and manually review the results. This is an excellent opportunity for someone who is curious about AI, enjoys solving proble…

JOB ID: 5840293769
TITLE: Data Analyst
COMPANY: TXP
SALARY: 0.0 - 124280.0
PREDICTED: 0

Data Analyst Location: Dunstable (2 days per week onsite) Duration: 3 Month Contract Start Date: ASAP Rate: £478 per day Inside IR35 Overview We are seeking an experienced Data Analyst to join a business-critical Martech transformation project. The successful candidate will play a key role in analysing, reco

In [323]:
df_jobs[df_jobs["salary_min"] == 0][
    [
        "job_id",
        "title",
        "company",
        "salary_min",
        "salary_max",
        "salary_is_predicted"
    ]
]

,job_id,title,company,salary_min,salary_max,salary_is_predicted
99,5829393076,Data Analyst,Red Recruitment,0.0,31200.0,0
122,5840293769,Data Analyst,TXP,0.0,124280.0,0
314,5826815795,Financial Reporting Analyst,Line Up Aviation,0.0,76980.0,0
387,5724380833,Senior Reporting and Insights Analyst,Clear Business,0.0,57000.0,0


In [324]:
(df_jobs["salary_min"] == 0).sum()

np.int64(4)

In [325]:
invalid_ranges = df_jobs[
    df_jobs["salary_min"] > df_jobs["salary_max"]
]

invalid_ranges[
    [
        "job_id",
        "title",
        "company",
        "salary_min",
        "salary_max"
    ]
]

,job_id,title,company,salary_min,salary_max


In [326]:
len(invalid_ranges)

0

In [327]:
daily_rate_pattern = r"\b(?:per day|per-day|daily rate|day rate|/day|p/d)\b"

daily_rate_jobs = df_jobs[
    df_jobs["description"]
    .str.contains(
        daily_rate_pattern,
        case=False,
        regex=True,
        na=False
    )
]

daily_rate_jobs[
    [
        "job_id",
        "title",
        "company",
        "salary_min",
        "salary_max",
        "description"
    ]
]

,job_id,title,company,salary_min,salary_max,description
114,5810959119,Data Analyst,Hays Technology,91000.00,104000.00,"DATA ANALYST 3-MONTH CONTRACT ON SITE - WARRINGTON £350.00 - £400.00 PER DAY (INSIDE) Your new role I am looking to recruit a Data Analyst to help drive smarter commercial decisions for our client on an initial 3-month contract basis. Sitting at the intersection of technology, operations and finance, you'll analyse transaction and performance data, build impactful dashboards, automate processes, and provide the insights that shape strategic decision-making. What you'll need to succeed The succe…"
121,5840340878,Data Analyst,TXP Technology x People,124280.00,124280.00,"Data Analyst Location: Dunstable (2 days per week onsite) Duration: 3 Month Contract Start Date: ASAP Rate: £478 per day Inside IR35 Overview We are seeking an experienced Data Analyst to join a business-critical Martech transformation project. The successful candidate will play a key role in analysing, reconciling, and consolidating large-scale customer data sets across multiple systems to support the migration into a new marketing technology environment. This is an exciting opportunity for a …"
122,5840293769,Data Analyst,TXP,0.00,124280.00,"Data Analyst Location: Dunstable (2 days per week onsite) Duration: 3 Month Contract Start Date: ASAP Rate: £478 per day Inside IR35 Overview We are seeking an experienced Data Analyst to join a business-critical Martech transformation project. The successful candidate will play a key role in analysing, reconciling, and consolidating large-scale customer data sets across multiple systems to support the migration into a new marketing technology environment. This is an exciting opportunity for a …"
140,5811535637,Data Analyst,HAYS,91000.00,104000.00,"DATA ANALYST 3-MONTH CONTRACT ON SITE - WARRINGTON £350.00 - £400.00 PER DAY (INSIDE) Your new role I am looking to recruit a Data Analyst to help drive smarter commercial decisions for our client on an initial 3-month contract basis. Sitting at the intersection of technology, operations and finance, you'll analyse transaction and performance data, build impactful dashboards, automate processes, and provide the insights that shape strategic decision-making. What you'll need to succeed The succe…"
146,5837138213,Data Analyst,Anson Mccade,60140.02,60140.02,"Data Analyst Data Standardisation Competitive Day Rate | Outside IR35 | Contract We are currently supporting a leading digital transformation consultancy on a Data Standardisation Discovery programme within the UK public sector . We are looking for an experienced Data Analyst to play a key role in understanding the current data landscape, identifying inconsistencies and data quality issues across multiple sources, and helping shape recommendations for future data standards and delivery. The Rol…"
150,5847339346,Data Analyst,Harnham - Data & Analytics Recruitment,91000.00,104000.00,"DATA ANALYST (CONTRACT) 6 MONTHS OUTSIDE IR35 £350 TO £400 P/D This is a high-impact contract opportunity for a hands-on Data Analyst to join a fast-moving growth initiative focused on scaling a local marketplace proposition. You will step into an existing work stream, ensuring continuity while bringing structure, pace, and insight to complex and imperfect data. The Company They are a well-established UK digital platform with millions of engaged members, known for delivering meaningful value th…"
154,5800473760,Data Analyst,Stott & May Professional Search Limited,104000.00,110500.00,"Data Analyst Location: Manchester, UK (Hybrid - Tuesday, Wednesday & Thursday Onsite) Day Rate: £425 per day (Inside IR35) Contract Duration: 12 Months Overview We are supporting a leading global organisation in their search for an experienced Data Analyst to join their European data function on a 12-month contract. This role will play a key part in supporting strategic business initiatives through data-driven insights, reporting, and analytics. Working 

## 2. Salary Data Quality Assessment

Salary fields were inspected for implausible values, incomplete ranges,
non-annual compensation and inconsistencies before transformation.


In [328]:
# Number of search-term relationships
print(job_search_terms.shape)

(532, 2)


In [329]:
# Zero lower bounds
(df_jobs["salary_min"] == 0).sum()

np.int64(4)

In [330]:
# Invalid ranges
invalid_ranges = df_jobs[
    df_jobs["salary_min"] > df_jobs["salary_max"]
]

len(invalid_ranges)

0

In [331]:
# Inspect anomalous salaries
for _, row in low_salary_jobs.iterrows():
    print("=" * 100)
    print("JOB ID:", row["job_id"])
    print("TITLE:", row["title"])
    print("COMPANY:", row["company"])
    print("SALARY:", row["salary_min"], "-", row["salary_max"])
    print(row["description"])

JOB ID: 5829393076
TITLE: Data Analyst
COMPANY: Red Recruitment
SALARY: 0.0 - 31200.0
Data Analyst Red Recruitment is recruiting a switched-on and detail-oriented Data Analyst to join an innovative fintech organisation in Bristol on a four-week temporary contract. Working within the Product team, you will help ensure the data used by an AI-powered platform is reliable and accurate. You will write and refine instructions for AI models, test data extractions and manually review the results. This is an excellent opportunity for someone who is curious about AI, enjoys solving proble…
JOB ID: 5840293769
TITLE: Data Analyst
COMPANY: TXP
SALARY: 0.0 - 124280.0
Data Analyst Location: Dunstable (2 days per week onsite) Duration: 3 Month Contract Start Date: ASAP Rate: £478 per day Inside IR35 Overview We are seeking an experienced Data Analyst to join a business-critical Martech transformation project. The successful candidate will play a key role in analysing, reconciling, and consolidating la

In [332]:
df_jobs["salary_min"] = df_jobs["salary_min"].replace(0, np.nan)
df_jobs["salary_max"] = df_jobs["salary_max"].replace(0, np.nan)

In [333]:
print("Zero salary_min:", (df_jobs["salary_min"] == 0).sum())
print("Zero salary_max:", (df_jobs["salary_max"] == 0).sum())

print("Missing salary_min:", df_jobs["salary_min"].isna().sum())
print("Missing salary_max:", df_jobs["salary_max"].isna().sum())

Zero salary_min: 0
Zero salary_max: 0
Missing salary_min: 4
Missing salary_max: 0


### 2.1 Zero Salary Bounds

Four vacancies contained a salary lower bound of zero. These values were
treated as missing rather than valid compensation because a zero salary does
not represent a plausible lower bound for the roles analysed.

In [334]:
# Replace invalid zero salary bounds with missing values
df_jobs["salary_min"] = df_jobs["salary_min"].replace(0, np.nan)
df_jobs["salary_max"] = df_jobs["salary_max"].replace(0, np.nan)

In [335]:
print("Missing salary_min:", df_jobs["salary_min"].isna().sum())
print("Missing salary_max:", df_jobs["salary_max"].isna().sum())

Missing salary_min: 4
Missing salary_max: 0


In [336]:
df_jobs["compensation_basis"] = "unknown"

In [337]:
daily_pattern = r"\b(?:per day|per-day|daily rate|day rate|/day|p/d)\b"

daily_mask = df_jobs["description"].str.contains(
    daily_pattern,
    case=False,
    regex=True,
    na=False
)

df_jobs.loc[daily_mask, "compensation_basis"] = "daily"

In [338]:
hourly_pattern = r"\b(?:per hour|hourly rate|hourly|/hour|p/h)\b"

hourly_mask = df_jobs["description"].str.contains(
    hourly_pattern,
    case=False,
    regex=True,
    na=False
)

df_jobs.loc[hourly_mask, "compensation_basis"] = "hourly"

In [339]:
annual_pattern = r"\b(?:per annum|per year|annual salary|annum|p/a)\b"

annual_mask = df_jobs["description"].str.contains(
    annual_pattern,
    case=False,
    regex=True,
    na=False
)

df_jobs.loc[
    annual_mask & ~daily_mask & ~hourly_mask,
    "compensation_basis"
] = "annual"

In [340]:
df_jobs["compensation_basis"].value_counts()

,count
compensation_basis,
unknown,441
annual,32
daily,14
hourly,4


In [341]:
df_jobs["mentions_daily_rate"] = daily_mask
df_jobs["mentions_hourly_rate"] = hourly_mask
df_jobs["mentions_annual_salary"] = annual_mask

In [342]:
df_jobs[
    df_jobs["mentions_daily_rate"] &
    df_jobs["mentions_hourly_rate"]
][
    [
        "job_id",
        "title",
        "company",
        "description"
    ]
]

,job_id,title,company,description
248,5840649326,BI and Insight Analyst,Opus People Solutions,"Business Insight & Intelligence Analyst Local Authority | 3-Month Temporary Contract Immediate Start Available Competitive Day Rate / Hourly Rate Are you passionate about transforming data into meaningful insight that drives real-world decisions? We are recruiting for a Business Insight & Intelligence Analyst to join a busy local authority team on a 3-month temporary contract. This is an exciting opportunity to play a key role in delivering intelligence, performance reporting and data-driven in…"


In [343]:
print("Daily mentions:", daily_mask.sum())
print("Hourly mentions:", hourly_mask.sum())
print("Annual mentions:", annual_mask.sum())

print(
    "Daily + Hourly:",
    (daily_mask & hourly_mask).sum()
)

print(
    "Daily + Annual:",
    (daily_mask & annual_mask).sum()
)

print(
    "Hourly + Annual:",
    (hourly_mask & annual_mask).sum()
)

Daily mentions: 15
Hourly mentions: 4
Annual mentions: 33
Daily + Hourly: 1
Daily + Annual: 1
Hourly + Annual: 0


In [344]:
df_jobs["mentions_daily_rate"] = daily_mask
df_jobs["mentions_hourly_rate"] = hourly_mask
df_jobs["mentions_annual_salary"] = annual_mask

In [345]:
df_jobs["compensation_basis_text"] = "unknown"

# Mixed cases first
df_jobs.loc[
    daily_mask & hourly_mask,
    "compensation_basis_text"
] = "daily_or_hourly"

df_jobs.loc[
    daily_mask & annual_mask,
    "compensation_basis_text"
] = "daily_and_annual"

df_jobs.loc[
    hourly_mask & annual_mask,
    "compensation_basis_text"
] = "hourly_and_annual"

# Single-basis cases
df_jobs.loc[
    daily_mask & ~hourly_mask & ~annual_mask,
    "compensation_basis_text"
] = "daily"

df_jobs.loc[
    hourly_mask & ~daily_mask & ~annual_mask,
    "compensation_basis_text"
] = "hourly"

df_jobs.loc[
    annual_mask & ~daily_mask & ~hourly_mask,
    "compensation_basis_text"
] = "annual"

In [346]:
df_jobs["compensation_basis_text"].value_counts()

,count
compensation_basis_text,
unknown,441
annual,32
daily,13
hourly,3
daily_or_hourly,1
daily_and_annual,1


In [347]:
daily_annual_overlap = df_jobs[
    daily_mask & annual_mask
][
    [
        "job_id",
        "title",
        "company",
        "salary_min",
        "salary_max",
        "description"
    ]
]

daily_annual_overlap

,job_id,title,company,salary_min,salary_max,description
405,5835661219,Asset Management Client Reporting Business Change Analyst,Atrium Workforce Solutions Ltd,148101.0,175500.0,"Contract Role – Asset Management Client Reporting Business Change Analyst – London/Hybrid – 5 months initial – Inside IR35 Role Overview: Job Title: Asset Management Client Reporting Business Change Analyst Location: London/Hybrid (4 days onsite per week) Contract Type: Contract Duration: 5 months Rate: £569.62 per day holidays (20 days 8 bank holidays per annum) PAYE OR £675 per day umbrella inside IR35 Deliver regulatory, technology-driven, and client-led change initiatives that support Cli…"


In [348]:
df_jobs["compensation_basis"]

,compensation_basis
0,unknown
1,unknown
2,unknown
3,unknown
4,unknown
...,...
486,daily
487,unknown
488,unknown
489,unknown


In [349]:
if "compensation_basis" in df_jobs.columns:
    df_jobs = df_jobs.drop(columns=["compensation_basis"])

In [350]:
[
    col
    for col in df_jobs.columns
    if "compensation" in col or "mentions_" in col
]

['mentions_daily_rate',
 'mentions_hourly_rate',
 'mentions_annual_salary',
 'compensation_basis_text']

In [351]:
daily_annual_overlap

,job_id,title,company,salary_min,salary_max,description
405,5835661219,Asset Management Client Reporting Business Change Analyst,Atrium Workforce Solutions Ltd,148101.0,175500.0,"Contract Role – Asset Management Client Reporting Business Change Analyst – London/Hybrid – 5 months initial – Inside IR35 Role Overview: Job Title: Asset Management Client Reporting Business Change Analyst Location: London/Hybrid (4 days onsite per week) Contract Type: Contract Duration: 5 months Rate: £569.62 per day holidays (20 days 8 bank holidays per annum) PAYE OR £675 per day umbrella inside IR35 Deliver regulatory, technology-driven, and client-led change initiatives that support Cli…"


In [352]:
df_jobs["compensation_basis_text"].value_counts()

,count
compensation_basis_text,
unknown,441
annual,32
daily,13
hourly,3
daily_or_hourly,1
daily_and_annual,1


In [353]:
annual_pattern = (
    r"(?:"
    r"\b(?:annual salary|annual pay)\b"
    r"|£\s?\d[\d,]*(?:\.\d+)?"
    r"(?:\s*(?:-|–|to)\s*£?\s?\d[\d,]*(?:\.\d+)?)?"
    r"\s*(?:per annum|per year|p/a)\b"
    r"|\b(?:salary|pay|compensation|package)\b.{0,50}"
    r"\b(?:per annum|per year|p/a)\b"
    r")"
)

annual_mask = df_jobs["description"].str.contains(
    annual_pattern,
    case=False,
    regex=True,
    na=False
)

In [354]:
df_jobs["mentions_daily_rate"] = daily_mask
df_jobs["mentions_hourly_rate"] = hourly_mask
df_jobs["mentions_annual_salary"] = annual_mask

In [355]:
df_jobs["compensation_basis_text"] = "unknown"

df_jobs.loc[
    daily_mask & hourly_mask,
    "compensation_basis_text"
] = "daily_or_hourly"

df_jobs.loc[
    daily_mask & ~hourly_mask & ~annual_mask,
    "compensation_basis_text"
] = "daily"

df_jobs.loc[
    hourly_mask & ~daily_mask & ~annual_mask,
    "compensation_basis_text"
] = "hourly"

df_jobs.loc[
    annual_mask & ~daily_mask & ~hourly_mask,
    "compensation_basis_text"
] = "annual"

In [356]:
print("Daily mentions:", daily_mask.sum())
print("Hourly mentions:", hourly_mask.sum())
print("Annual mentions:", annual_mask.sum())

print("Daily + Hourly:", (daily_mask & hourly_mask).sum())
print("Daily + Annual:", (daily_mask & annual_mask).sum())
print("Hourly + Annual:", (hourly_mask & annual_mask).sum())

df_jobs["compensation_basis_text"].value_counts()

Daily mentions: 15
Hourly mentions: 4
Annual mentions: 31
Daily + Hourly: 1
Daily + Annual: 0
Hourly + Annual: 0


,count
compensation_basis_text,
unknown,442
annual,31
daily,14
hourly,3
daily_or_hourly,1


## 3. Potential Duplicate Advertisements

Exact Adzuna job IDs were already deduplicated. However, the same vacancy may
be syndicated or reposted under different job IDs.

Potential duplicates are therefore assessed using normalized job title,
location and job-description text before any records are removed.

In [357]:
import re
import html
import unicodedata

In [358]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [359]:
df_jobs["title_norm"] = df_jobs["title"].apply(normalize_text)

df_jobs["location_norm"] = df_jobs["location"].apply(normalize_text)

df_jobs["description_norm"] = (
    df_jobs["description"].apply(normalize_text)
)

In [360]:
df_jobs[
    [
        "title",
        "title_norm",
        "location",
        "location_norm"
    ]
].head()

,title,title_norm,location,location_norm
0,Lead Data Analyst/Project Controller,lead data analyst project controller,"Padiham, Burnley",padiham burnley
1,Lead Data Analyst/Project Controller,lead data analyst project controller,"Penwortham, Preston",penwortham preston
2,Lead Data Analyst/Project Controller,lead data analyst project controller,"Samlesbury, Preston",samlesbury preston
3,Data Analyst,data analyst,"Antrim, County Antrim",antrim county antrim
4,Data Analyst,data analyst,"Banbridge, County Down",banbridge county down


In [361]:
strong_duplicate_mask = df_jobs.duplicated(
    subset=[
        "title_norm",
        "location_norm",
        "description_norm"
    ],
    keep=False
)

strong_duplicate_candidates = (
    df_jobs[strong_duplicate_mask]
    .sort_values(
        [
            "title_norm",
            "location_norm",
            "description_norm"
        ]
    )
)

print(
    "Rows involved in strong duplicate candidates:",
    len(strong_duplicate_candidates)
)

Rows involved in strong duplicate candidates: 12


In [362]:
strong_duplicate_candidates[
    [
        "job_id",
        "title",
        "company",
        "location",
        "salary_min",
        "salary_max",
        "created"
    ]
]

,job_id,title,company,location,salary_min,salary_max,created
444,5825674495,BI & Data Analyst,Hays Technology,"Merseyside, North West England",35000.00,38000.00,2026-08-02 13:03:01+00:00
446,5825728077,BI & Data Analyst,Hays Specialist Recruitment Limited,"Merseyside, North West England",35000.00,38000.00,2026-08-02 15:07:27+00:00
369,5830850030,Business and Data Reporting Analyst 12-month FTC,Norton Rose Fulbright LLP,"Newcastle Upon Tyne, Tyne & Wear",37790.21,37790.21,2026-08-06 19:54:25+00:00
370,5815352878,Business and Data Reporting Analyst (12-month FTC),NRF,"Newcastle Upon Tyne, Tyne & Wear",41200.33,41200.33,2026-07-25 12:13:47+00:00
12,5829374055,Data Analyst,red recruitment,"Bristol, South West England",31200.00,31200.00,2026-08-05 19:21:02+00:00
117,5829135139,Data Analyst,Red Recruitment,"Bristol, South West England",31200.00,31200.00,2026-08-05 15:16:57+00:00
109,5679132312,Data Analyst,G-Research,"London, UK",56635.39,56635.39,2026-03-26 10:09:27+00:00
194,5817419816,Data Analyst,G-Research,"London, UK",51993.86,51993.86,2026-07-27 14:39:20+00:00
309,5842137733,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00,2026-08-14 21:01:34+00:00
320,5841817050,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00,2026-08-14 13:26:24+00:00


In [363]:
description_duplicate_mask = df_jobs.duplicated(
    subset=["description_norm"],
    keep=False
)

description_duplicates = (
    df_jobs[description_duplicate_mask]
    .sort_values("description_norm")
)

print(
    "Rows sharing an identical normalized description:",
    len(description_duplicates)
)

Rows sharing an identical normalized description: 187


In [364]:
description_duplicates[
    [
        "job_id",
        "title",
        "company",
        "location",
        "salary_min",
        "salary_max"
    ]
].head(30)

,job_id,title,company,location,salary_min,salary_max
279,5846664611,"Trainee Data Analyst (Excel, SQL & Power BI)",Netcom Online Learning,"Sheffield, South Yorkshire",30758.85,30758.85
287,5846667330,Trainee Data Analyst,Netcom Online Learning,"Sheffield, South Yorkshire",29552.10,29552.10
278,5733579840,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon UK Services Ltd.,"London, UK",55144.76,55144.76
283,5840373713,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon,"London, UK",60820.68,60820.68
359,5830781649,Senior Financial Planning and Reporting Analyst,Marc Daniels,"Englefield, Reading",100000.00,110000.00
361,5830849183,Senior Financial Planning and Reporting Analyst,Marc Daniels,"Reading, Berkshire",100000.00,110000.00
463,5720676258,Demand Planner,Potential Recruitment,"Trafford Park, Manchester",33000.00,35000.00
431,5848138269,Demand Planner,Potential Recruitment,"Manchester, Greater Manchester",35000.00,35000.00
28,5848124437,Data Analyst Trainee,ITOL Recruit,"Romford, Essex",30000.00,50000.00
27,5835558668,Data Analyst Trainee,ITOL Recruit,"Wolverhampton, West Midlands",30000.00,50000.00


In [365]:
df_jobs["strong_duplicate_group"] = (
    df_jobs
    .groupby(
        [
            "title_norm",
            "location_norm",
            "description_norm"
        ],
        sort=False
    )
    .ngroup()
)

In [366]:
group_sizes = (
    df_jobs
    .groupby("strong_duplicate_group")
    .size()
)

df_jobs["strong_duplicate_group_size"] = (
    df_jobs["strong_duplicate_group"]
    .map(group_sizes)
)

In [367]:
df_jobs[
    df_jobs["strong_duplicate_group_size"] > 1
][
    [
        "strong_duplicate_group",
        "job_id",
        "title",
        "company",
        "location",
        "salary_min",
        "salary_max"
    ]
].sort_values("strong_duplicate_group")

,strong_duplicate_group,job_id,title,company,location,salary_min,salary_max
12,12,5829374055,Data Analyst,red recruitment,"Bristol, South West England",31200.00,31200.00
117,12,5829135139,Data Analyst,Red Recruitment,"Bristol, South West England",31200.00,31200.00
109,109,5679132312,Data Analyst,G-Research,"London, UK",56635.39,56635.39
194,109,5817419816,Data Analyst,G-Research,"London, UK",51993.86,51993.86
278,276,5733579840,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon UK Services Ltd.,"London, UK",55144.76,55144.76
283,276,5840373713,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon,"London, UK",60820.68,60820.68
309,306,5842137733,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00
320,306,5841817050,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00
369,365,5830850030,Business and Data Reporting Analyst 12-month FTC,Norton Rose Fulbright LLP,"Newcastle Upon Tyne, Tyne & Wear",37790.21,37790.21
370,365,5815352878,Business and Data Reporting Analyst (12-month FTC),NRF,"Newcastle Upon Tyne, Tyne & Wear",41200.33,41200.33


In [368]:
len(strong_duplicate_candidates)

12

In [369]:
strong_duplicate_candidates[
    [
        "job_id",
        "title",
        "company",
        "location",
        "salary_min",
        "salary_max"
    ]
]

,job_id,title,company,location,salary_min,salary_max
444,5825674495,BI & Data Analyst,Hays Technology,"Merseyside, North West England",35000.00,38000.00
446,5825728077,BI & Data Analyst,Hays Specialist Recruitment Limited,"Merseyside, North West England",35000.00,38000.00
369,5830850030,Business and Data Reporting Analyst 12-month FTC,Norton Rose Fulbright LLP,"Newcastle Upon Tyne, Tyne & Wear",37790.21,37790.21
370,5815352878,Business and Data Reporting Analyst (12-month FTC),NRF,"Newcastle Upon Tyne, Tyne & Wear",41200.33,41200.33
12,5829374055,Data Analyst,red recruitment,"Bristol, South West England",31200.00,31200.00
117,5829135139,Data Analyst,Red Recruitment,"Bristol, South West England",31200.00,31200.00
109,5679132312,Data Analyst,G-Research,"London, UK",56635.39,56635.39
194,5817419816,Data Analyst,G-Research,"London, UK",51993.86,51993.86
309,5842137733,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00
320,5841817050,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00


In [370]:
len(description_duplicates)

187

In [371]:
description_duplicates[
    [
        "job_id",
        "title",
        "company",
        "location"
    ]
].head(30)

,job_id,title,company,location
279,5846664611,"Trainee Data Analyst (Excel, SQL & Power BI)",Netcom Online Learning,"Sheffield, South Yorkshire"
287,5846667330,Trainee Data Analyst,Netcom Online Learning,"Sheffield, South Yorkshire"
278,5733579840,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon UK Services Ltd.,"London, UK"
283,5840373713,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon,"London, UK"
359,5830781649,Senior Financial Planning and Reporting Analyst,Marc Daniels,"Englefield, Reading"
361,5830849183,Senior Financial Planning and Reporting Analyst,Marc Daniels,"Reading, Berkshire"
463,5720676258,Demand Planner,Potential Recruitment,"Trafford Park, Manchester"
431,5848138269,Demand Planner,Potential Recruitment,"Manchester, Greater Manchester"
28,5848124437,Data Analyst Trainee,ITOL Recruit,"Romford, Essex"
27,5835558668,Data Analyst Trainee,ITOL Recruit,"Wolverhampton, West Midlands"


In [372]:
strong_duplicate_review = df_jobs[
    df_jobs["strong_duplicate_group_size"] > 1
][
    [
        "strong_duplicate_group",
        "job_id",
        "title",
        "company",
        "location",
        "salary_min",
        "salary_max",
        "salary_is_predicted",
        "salary_source",
        "created"
    ]
].sort_values(
    ["strong_duplicate_group", "created"]
)

strong_duplicate_review

,strong_duplicate_group,job_id,title,company,location,salary_min,salary_max,salary_is_predicted,salary_source,created
117,12,5829135139,Data Analyst,Red Recruitment,"Bristol, South West England",31200.00,31200.00,0,Advertised / non-predicted,2026-08-05 15:16:57+00:00
12,12,5829374055,Data Analyst,red recruitment,"Bristol, South West England",31200.00,31200.00,0,Advertised / non-predicted,2026-08-05 19:21:02+00:00
109,109,5679132312,Data Analyst,G-Research,"London, UK",56635.39,56635.39,1,Adzuna predicted,2026-03-26 10:09:27+00:00
194,109,5817419816,Data Analyst,G-Research,"London, UK",51993.86,51993.86,1,Adzuna predicted,2026-07-27 14:39:20+00:00
278,276,5733579840,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon UK Services Ltd.,"London, UK",55144.76,55144.76,1,Adzuna predicted,2026-05-19 01:51:49+00:00
283,276,5840373713,"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",Amazon,"London, UK",60820.68,60820.68,1,Adzuna predicted,2026-08-13 16:13:11+00:00
320,306,5841817050,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00,0,Advertised / non-predicted,2026-08-14 13:26:24+00:00
309,306,5842137733,Data & Reporting Analyst,Robert Half,"Slough, Berkshire",30000.00,40000.00,0,Advertised / non-predicted,2026-08-14 21:01:34+00:00
370,365,5815352878,Business and Data Reporting Analyst (12-month FTC),NRF,"Newcastle Upon Tyne, Tyne & Wear",41200.33,41200.33,1,Adzuna predicted,2026-07-25 12:13:47+00:00
369,365,5830850030,Business and Data Reporting Analyst 12-month FTC,Norton Rose Fulbright LLP,"Newcastle Upon Tyne, Tyne & Wear",37790.21,37790.21,1,Adzuna predicted,2026-08-06 19:54:25+00:00


In [373]:
duplicate_group_summary = (
    strong_duplicate_review
    .groupby("strong_duplicate_group")
    .agg(
        records=("job_id", "size"),
        first_posted=("created", "min"),
        last_posted=("created", "max"),
        companies=("company", lambda x: list(x)),
        salary_mins=("salary_min", lambda x: list(x)),
        salary_maxs=("salary_max", lambda x: list(x)),
        salary_sources=("salary_source", lambda x: list(x))
    )
)

duplicate_group_summary["days_apart"] = (
    duplicate_group_summary["last_posted"]
    - duplicate_group_summary["first_posted"]
).dt.total_seconds() / 86400

duplicate_group_summary

,records,first_posted,last_posted,companies,salary_mins,salary_maxs,salary_sources,days_apart
strong_duplicate_group,,,,,,,,
12,2,2026-08-05 15:16:57+00:00,2026-08-05 19:21:02+00:00,"[Red Recruitment, red recruitment]","[31200.0, 31200.0]","[31200.0, 31200.0]","[Advertised / non-predicted, Advertised / non-predicted]",0.169502
109,2,2026-03-26 10:09:27+00:00,2026-07-27 14:39:20+00:00,"[G-Research, G-Research]","[56635.39, 51993.86]","[56635.39, 51993.86]","[Adzuna predicted, Adzuna predicted]",123.187419
276,2,2026-05-19 01:51:49+00:00,2026-08-13 16:13:11+00:00,"[Amazon UK Services Ltd., Amazon]","[55144.76, 60820.68]","[55144.76, 60820.68]","[Adzuna predicted, Adzuna predicted]",86.598171
306,2,2026-08-14 13:26:24+00:00,2026-08-14 21:01:34+00:00,"[Robert Half, Robert Half]","[30000.0, 30000.0]","[40000.0, 40000.0]","[Advertised / non-predicted, Advertised / non-predicted]",0.316088
365,2,2026-07-25 12:13:47+00:00,2026-08-06 19:54:25+00:00,"[NRF, Norton Rose Fulbright LLP]","[41200.33, 37790.21]","[41200.33, 37790.21]","[Adzuna predicted, Adzuna predicted]",12.319884
439,2,2026-08-02 13:03:01+00:00,2026-08-02 15:07:27+00:00,"[Hays Technology, Hays Specialist Recruitment Limited]","[35000.0, 35000.0]","[38000.0, 38000.0]","[Advertised / non-predicted, Advertised / non-predicted]",0.086412


In [374]:
df_jobs["duplicate_status"] = "unique"

In [375]:
high_confidence_groups = [12, 306, 439]
probable_repost_groups = [365]
repeat_opening_groups = [109, 276]

df_jobs.loc[
    df_jobs["strong_duplicate_group"].isin(high_confidence_groups),
    "duplicate_status"
] = "high_confidence_duplicate"

df_jobs.loc[
    df_jobs["strong_duplicate_group"].isin(probable_repost_groups),
    "duplicate_status"
] = "probable_repost"

df_jobs.loc[
    df_jobs["strong_duplicate_group"].isin(repeat_opening_groups),
    "duplicate_status"
] = "possible_repeat_opening"

In [376]:
df_jobs["duplicate_status"].value_counts()

,count
duplicate_status,
unique,479
high_confidence_duplicate,6
possible_repeat_opening,4
probable_repost,2


In [377]:
df_jobs["keep_record"] = True

In [378]:
dedup_groups = (
    high_confidence_groups
    + probable_repost_groups
)

In [379]:
for group in dedup_groups:

    group_rows = df_jobs[
        df_jobs["strong_duplicate_group"] == group
    ]

    latest_index = group_rows["created"].idxmax()

    df_jobs.loc[
        group_rows.index,
        "keep_record"
    ] = False

    df_jobs.loc[
        latest_index,
        "keep_record"
    ] = True

In [380]:
df_jobs[
    df_jobs["strong_duplicate_group"].isin(dedup_groups)
][
    [
        "strong_duplicate_group",
        "job_id",
        "company",
        "created",
        "duplicate_status",
        "keep_record"
    ]
].sort_values(
    ["strong_duplicate_group", "created"]
)

,strong_duplicate_group,job_id,company,created,duplicate_status,keep_record
117,12,5829135139,Red Recruitment,2026-08-05 15:16:57+00:00,high_confidence_duplicate,False
12,12,5829374055,red recruitment,2026-08-05 19:21:02+00:00,high_confidence_duplicate,True
320,306,5841817050,Robert Half,2026-08-14 13:26:24+00:00,high_confidence_duplicate,False
309,306,5842137733,Robert Half,2026-08-14 21:01:34+00:00,high_confidence_duplicate,True
370,365,5815352878,NRF,2026-07-25 12:13:47+00:00,probable_repost,False
369,365,5830850030,Norton Rose Fulbright LLP,2026-08-06 19:54:25+00:00,probable_repost,True
444,439,5825674495,Hays Technology,2026-08-02 13:03:01+00:00,high_confidence_duplicate,False
446,439,5825728077,Hays Specialist Recruitment Limited,2026-08-02 15:07:27+00:00,high_confidence_duplicate,True


In [381]:
df_jobs_clean = (
    df_jobs[
        df_jobs["keep_record"]
    ]
    .reset_index(drop=True)
    .copy()
)

In [382]:
print("Before deduplication:", len(df_jobs))
print("After deduplication:", len(df_jobs_clean))
print("Records removed:", len(df_jobs) - len(df_jobs_clean))

Before deduplication: 491
After deduplication: 487
Records removed: 4


In [383]:
df_jobs_clean["job_id"].nunique() == len(df_jobs_clean)

True

In [384]:
df_jobs_clean.duplicated(
    subset=[
        "title_norm",
        "location_norm",
        "description_norm"
    ]
).sum()

np.int64(2)

### 3.1 Duplicate Resolution Strategy

Potential duplicate advertisements were classified using a conservative,
rule-based approach.

Records with identical normalized title, location and description posted
within approximately one day and showing matching salary information were
classified as high-confidence duplicates.

A matching advertisement reposted within approximately two weeks was treated
as a probable repost when supporting evidence indicated the same vacancy.

Records separated by several months were retained as possible repeat openings,
as identical job descriptions may be reused when an employer recruits for the
same role again.

For duplicate and probable-repost groups, the most recently posted record was
retained as the canonical observation.

This approach avoids inflating vacancy counts while reducing the risk of
incorrectly removing legitimate repeat hiring.

## 4. Job Relevance & Title Standardisation

Job-search queries may return vacancies outside the intended analytics
occupation scope. Job titles are therefore profiled and classified before
market-level statistics are calculated.

Original job titles are preserved, while derived fields are created for
analytical grouping and relevance assessment.

In [385]:
print("Unique job titles:", df_jobs_clean["title"].nunique())

Unique job titles: 210


In [386]:
df_jobs_clean["title"].value_counts().head(40)

,count
title,
Data Analyst,108
Data Analyst Trainee,58
Trainee Data Analyst,20
Business Intelligence Analyst,12
Data Analyst Placement Programme No Experience Needed,11
Financial Reporting Analyst,8
Data Science Trainee,4
Regulatory Reporting Analyst,4
Lead Data Analyst/Project Controller,4


In [387]:
sorted(
    df_jobs_clean["title"]
    .dropna()
    .unique()
)[:100]

['AR Analyst',
 'Account Manager/ Content Producer -Niche B2B Tech Agency',
 'Accounts Receivable - Lead Finance Analyst - AC6',
 'Accounts Receivable and Payable Reporting Analyst',
 'Advanced Specialist, Financial Reporting, Planning and Analysis',
 'Analyst',
 'Analyst O&T Accounting',
 'Analyst Relations Assistant Manager',
 'Analyst – Mandarin / Cantonese',
 'Asbestos Site Analyst',
 'Asset Management Client Reporting Business Change Analyst',
 'Assistant Manager - Business Analyst - Regulatory Reporting-BFS031374',
 'BI & Data Analyst',
 'BI & Reporting Analyst',
 'BI Analyst',
 'BI BA – Business Analyst with Business Intelligence (IT)',
 'BI Developer',
 'BI and Insight Analyst',
 'Basel regulatory reporting senior business analyst',
 'Business Analyst',
 'Business Analyst - Regulatory Change & Transaction Reporting',
 'Business Analyst - Salesforce System',
 'Business Analyst - Security & Intelligence',
 'Business Analyst - Security and Intelligence',
 'Business Analyst – Regul

In [388]:
df_jobs_clean["title_clean"] = (
    df_jobs_clean["title"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [389]:
df_jobs_clean[
    ["title", "title_clean"]
].head(15)

,title,title_clean
0,Lead Data Analyst/Project Controller,lead data analyst project controller
1,Lead Data Analyst/Project Controller,lead data analyst project controller
2,Lead Data Analyst/Project Controller,lead data analyst project controller
3,Data Analyst,data analyst
4,Data Analyst,data analyst
5,Data Analyst,data analyst
6,Master Data Analyst,master data analyst
7,Data Analyst - Manufacturing,data analyst manufacturing
8,Lead Data Analyst/Project Controller,lead data analyst project controller
9,Data Analyst Placement Programme No Experience Needed,data analyst placement programme no experience needed


In [390]:
def classify_job_family(title):

    if pd.isna(title):
        return "Unknown"

    title = title.lower()

    if "data analyst" in title:
        return "Data Analyst"

    if (
        "business intelligence" in title
        or "bi analyst" in title
        or "bi &" in title
        or "bi and" in title
    ):
        return "BI Analyst"

    if "reporting analyst" in title:
        return "Reporting Analyst"

    if (
        "insight analyst" in title
        or "insights analyst" in title
    ):
        return "Insights Analyst"

    if "product analyst" in title:
        return "Product Analyst"

    return "Other"

In [391]:
df_jobs_clean["job_family"] = (
    df_jobs_clean["title"]
    .apply(classify_job_family)
)

In [392]:
df_jobs_clean["job_family"].value_counts()

,count
job_family,
Data Analyst,231
Other,141
Reporting Analyst,64
BI Analyst,47
Insights Analyst,4


In [393]:
other_titles = (
    df_jobs_clean[
        df_jobs_clean["job_family"] == "Other"
    ]["title"]
    .value_counts()
)

other_titles.head(60)

,count
title,
Data Science Trainee,4
Record to Report Ops Analyst - Record To Report,4
Finance Analyst,4
"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",3
Finance/Business Analyst Internship,3
Data Business Analyst,3
Senior Planner,3
BI Developer,3
Business Analyst - Salesforce System,2


In [394]:
sorted(
    df_jobs_clean.loc[
        df_jobs_clean["job_family"] == "Other",
        "title"
    ]
    .dropna()
    .unique()
)[:150]

['AR Analyst',
 'Account Manager/ Content Producer -Niche B2B Tech Agency',
 'Accounts Receivable - Lead Finance Analyst - AC6',
 'Advanced Specialist, Financial Reporting, Planning and Analysis',
 'Analyst',
 'Analyst O&T Accounting',
 'Analyst Relations Assistant Manager',
 'Analyst – Mandarin / Cantonese',
 'Asbestos Site Analyst',
 'Asset Management Client Reporting Business Change Analyst',
 'Assistant Manager - Business Analyst - Regulatory Reporting-BFS031374',
 'BI Developer',
 'Basel regulatory reporting senior business analyst',
 'Business Analyst',
 'Business Analyst - Regulatory Change & Transaction Reporting',
 'Business Analyst - Salesforce System',
 'Business Analyst - Security & Intelligence',
 'Business Analyst - Security and Intelligence',
 'Business Analyst – Regulatory Change & Transaction Reporting',
 'Business Planning Analyst',
 'Client Tax Reporting Lead Analyst 2',
 'Commercial Analyst',
 'Contract Senior Analyst - Record To Report',
 'Controllers, Non-Financia

In [395]:
df_jobs_clean["title"].nunique()

210

In [396]:
df_jobs_clean["job_family"].value_counts()

,count
job_family,
Data Analyst,231
Other,141
Reporting Analyst,64
BI Analyst,47
Insights Analyst,4


In [397]:
other_titles.head(60)

,count
title,
Data Science Trainee,4
Record to Report Ops Analyst - Record To Report,4
Finance Analyst,4
"Senior Product & Customer Insights Manager, Customer Experience and Business Trends",3
Finance/Business Analyst Internship,3
Data Business Analyst,3
Senior Planner,3
BI Developer,3
Business Analyst - Salesforce System,2


### 4.1 Training and Placement Programme Assessment

Some search results appear to advertise training, placement or career-change
programmes rather than conventional employment vacancies.

These records are assessed separately before occupational relevance is
classified to prevent training advertisements from inflating vacancy counts.

In [398]:
programme_pattern = (
    r"\b(?:"
    r"trainee|training course|training programme|"
    r"placement programme|no experience needed|"
    r"career programme|career program|bootcamp"
    r")\b"
)

programme_mask = (
    df_jobs_clean["title"]
    .str.contains(
        programme_pattern,
        case=False,
        regex=True,
        na=False
    )
)

print(
    "Potential training/programme records:",
    programme_mask.sum()
)

Potential training/programme records: 97


In [399]:
df_jobs_clean.loc[
    programme_mask,
    "title"
].value_counts()

,count
title,
Data Analyst Trainee,58
Trainee Data Analyst,20
Data Analyst Placement Programme No Experience Needed,11
Data Science Trainee,4
Data Analyst Placement Programme,1
Trainee Data Analyst No experience needed,1
"Trainee Data Analyst (Excel, SQL & Power BI)",1
"Data Analyst Training Course (Excel, SQL & Power BI)",1


In [400]:
df_jobs_clean.loc[
    programme_mask,
    "company"
].value_counts()

,count
company,
ITOL Recruit,80
IT Online Learning,13
Netcom Online Learning,2
Recruitment Solutions,1
Netcom Training,1


In [401]:
programme_review = df_jobs_clean.loc[
    programme_mask,
    [
        "job_id",
        "title",
        "company",
        "location",
        "salary_min",
        "salary_max",
        "description"
    ]
].copy()

programme_review[
    [
        "job_id",
        "title",
        "company",
        "location"
    ]
].head(40)

,job_id,title,company,location
9,5831512580,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Cardiff County, Wales"
10,5843949911,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Bristol, South West England"
16,5830868210,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Manchester, Greater Manchester"
17,5836182466,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Liverpool, Merseyside"
18,5831512570,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Leeds, West Yorkshire"
19,5832939296,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Birmingham, West Midlands"
20,5831302313,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Edinburgh, Scotland"
22,5830015735,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Sheffield, South Yorkshire"
23,5830015742,Data Analyst Placement Programme No Experience Needed,IT Online Learning,"Glasgow, Scotland"
24,5837973168,Data Analyst Trainee,ITOL Recruit,"Cardiff County, Wales"


In [402]:
sample_programmes = (
    programme_review
    .drop_duplicates(
        subset=["title", "company"]
    )
)

for _, row in sample_programmes.iterrows():

    print("=" * 100)
    print("TITLE:", row["title"])
    print("COMPANY:", row["company"])
    print("LOCATION:", row["location"])
    print()

    print(row["description"][:1200])
    print()

TITLE: Data Analyst Placement Programme No Experience Needed
COMPANY: IT Online Learning
LOCATION: Cardiff County, Wales

Data Analyst Placement Programme no experience needed From £30,000 to £65,000 per annum Trainee Data Analyst £30,000£65,000 Job Programme This is a self-funded programme that leads to employment, fees apply. Job Guarantee Complete the programme and get a job, or get your course fees back. Location: UK Wide (Remote Opportunities Available) Looking to start a career in Data Analysis? We are offering a structured pathway into Data Analysis, designed to help you enter the industry with no prior exp…

TITLE: Data Analyst Trainee
COMPANY: ITOL Recruit
LOCATION: Cardiff County, Wales

Are you looking to benefit from a new career in Data Analysis? If you are detail orientated, perceptive, organised, competent, analytical and can communicate well with those around you; you could have a truly rewarding future as a Data Analyst Please note this career program is designed for e

In [403]:
programme_mask.sum()

np.int64(97)

In [404]:
df_jobs_clean.loc[
    programme_mask,
    "title"
].value_counts()

,count
title,
Data Analyst Trainee,58
Trainee Data Analyst,20
Data Analyst Placement Programme No Experience Needed,11
Data Science Trainee,4
Data Analyst Placement Programme,1
Trainee Data Analyst No experience needed,1
"Trainee Data Analyst (Excel, SQL & Power BI)",1
"Data Analyst Training Course (Excel, SQL & Power BI)",1


In [405]:
df_jobs_clean.loc[
    programme_mask,
    "company"
].value_counts()

,count
company,
ITOL Recruit,80
IT Online Learning,13
Netcom Online Learning,2
Recruitment Solutions,1
Netcom Training,1


### 4.2 Confirmed Training Programme Advertisements

Potential training-related records were manually reviewed using representative
job descriptions.

Advertisements explicitly offering self-funded or funded courses, career
programmes, job guarantees, training pathways or course-based preparation for
future employment were classified as training advertisements rather than
vacancies.

A genuine trainee vacancy identified during review was retained, demonstrating
why title-based exclusion alone was insufficient.

In [406]:
training_ad_pattern = (
    r"(?:"
    r"self[- ]funded programme|"
    r"self[- ]funded training|"
    r"fees apply|"
    r"course fees|"
    r"job guarantee|"
    r"career programme|"
    r"career program|"
    r"training programme|"
    r"training program|"
    r"training course|"
    r"fully[- ]funded data course|"
    r"train online|"
    r"online training|"
    r"learners have successfully|"
    r"certificate in data|"

    # ITOL template sometimes truncated before
    # the explicit 'career programme' wording
    r"are you looking to benefit from a new career in data analysis"
    r")"
)

confirmed_training_mask = (
    programme_mask
    &
    df_jobs_clean["description"].str.contains(
        training_ad_pattern,
        case=False,
        regex=True,
        na=False
    )
)

print(
    "Potential programme records:",
    programme_mask.sum()
)

print(
    "Confirmed training advertisements:",
    confirmed_training_mask.sum()
)

print(
    "Potential programme records retained as vacancies:",
    (programme_mask & ~confirmed_training_mask).sum()
)

Potential programme records: 97
Confirmed training advertisements: 96
Potential programme records retained as vacancies: 1


In [407]:
confirmed_training_mask = (
    programme_mask
    &
    df_jobs_clean["description"].str.contains(
        training_ad_pattern,
        case=False,
        regex=True,
        na=False
    )
)

In [408]:
print(
    "Potential programme records:",
    programme_mask.sum()
)

print(
    "Confirmed training advertisements:",
    confirmed_training_mask.sum()
)

print(
    "Potential programme records retained as vacancies:",
    (programme_mask & ~confirmed_training_mask).sum()
)

Potential programme records: 97
Confirmed training advertisements: 96
Potential programme records retained as vacancies: 1


In [409]:
df_jobs_clean.loc[
    programme_mask & ~confirmed_training_mask,
    [
        "job_id",
        "title",
        "company",
        "location",
        "description"
    ]
]

,job_id,title,company,location,description
79,5819806247,Trainee Data Analyst,Recruitment Solutions,"Westerham, Kent","Trainee Data Analyst Are you… an individual with a love for data analysis and looking for a new challenge? Do you… want a role where you can as an essential part of the team? Can you… work under time constraints and create and present reports? If so… Read on and apply today! About the Role We are looking for an analytical individual to join an established business based in Westerham. This is an amazing opportunity to join an established and international business, to build your skills and becom…"


In [410]:
df_jobs_clean["relevance_status"] = "review"

df_jobs_clean.loc[
    confirmed_training_mask,
    "relevance_status"
] = "exclude_training_programme"

In [411]:
df_jobs_clean["relevance_status"].value_counts()

,count
relevance_status,
review,391
exclude_training_programme,96


In [412]:
training_share = (
    confirmed_training_mask.sum()
    / len(df_jobs_clean)
    * 100
)

print(
    f"Training advertisements: {training_share:.1f}%"
)

Training advertisements: 19.7%


In [413]:
confirmed_training_mask.sum()

np.int64(96)

In [414]:
df_jobs_clean.loc[
    programme_mask & ~confirmed_training_mask,
    ["title", "company", "location"]
]

,title,company,location
79,Trainee Data Analyst,Recruitment Solutions,"Westerham, Kent"


In [415]:
df_jobs_clean["relevance_status"].value_counts()

,count
relevance_status,
review,391
exclude_training_programme,96


In [416]:
print(f"Training share: {training_share:.1f}%")

Training share: 19.7%


In [417]:
df_jobs_clean.loc[
    programme_mask & ~confirmed_training_mask,
    [
        "job_id",
        "title",
        "company",
        "location"
    ]
]

,job_id,title,company,location
79,5819806247,Trainee Data Analyst,Recruitment Solutions,"Westerham, Kent"


In [418]:
df_vacancies = (
    df_jobs_clean[
        df_jobs_clean["relevance_status"]
        != "exclude_training_programme"
    ]
    .reset_index(drop=True)
    .copy()
)

In [419]:
print("Deduplicated advertisements:", len(df_jobs_clean))
print("Training/programmes excluded:", confirmed_training_mask.sum())
print("Vacancy candidates remaining:", len(df_vacancies))

Deduplicated advertisements: 487
Training/programmes excluded: 96
Vacancy candidates remaining: 391


### 4.3 Occupational Relevance Assessment

After removing confirmed training and career-programme advertisements,
the remaining vacancy candidates were assessed for occupational relevance.

Roles were separated into:

- **Core Analytics** — roles directly aligned with data analysis, business
  intelligence, reporting, insights or closely related analytical work.
- **Adjacent Analytics** — roles with substantial analytical responsibilities
  but belonging primarily to another occupational family.
- **Exclude** — roles outside the intended Data Analytics labour-market scope.
- **Review** — ambiguous titles requiring further inspection.

A conservative classification approach was used to avoid inflating the
Data Analyst market with unrelated roles.

In [420]:
print("Vacancy candidates:", len(df_vacancies))
print("Unique titles:", df_vacancies["title"].nunique())

Vacancy candidates: 391
Unique titles: 203


In [421]:
df_vacancies["title"].value_counts().head(50)

,count
title,
Data Analyst,108
Business Intelligence Analyst,12
Financial Reporting Analyst,8
Lead Data Analyst/Project Controller,4
Regulatory Reporting Analyst,4
Finance Analyst,4
Record to Report Ops Analyst - Record To Report,4
BI Developer,3
BI Analyst,3


In [422]:
sorted(
    df_vacancies["title"]
    .dropna()
    .unique()
)[:150]

['AR Analyst',
 'Account Manager/ Content Producer -Niche B2B Tech Agency',
 'Accounts Receivable - Lead Finance Analyst - AC6',
 'Accounts Receivable and Payable Reporting Analyst',
 'Advanced Specialist, Financial Reporting, Planning and Analysis',
 'Analyst',
 'Analyst O&T Accounting',
 'Analyst Relations Assistant Manager',
 'Analyst – Mandarin / Cantonese',
 'Asbestos Site Analyst',
 'Asset Management Client Reporting Business Change Analyst',
 'Assistant Manager - Business Analyst - Regulatory Reporting-BFS031374',
 'BI & Data Analyst',
 'BI & Reporting Analyst',
 'BI Analyst',
 'BI BA – Business Analyst with Business Intelligence (IT)',
 'BI Developer',
 'BI and Insight Analyst',
 'Basel regulatory reporting senior business analyst',
 'Business Analyst',
 'Business Analyst - Regulatory Change & Transaction Reporting',
 'Business Analyst - Salesforce System',
 'Business Analyst - Security & Intelligence',
 'Business Analyst - Security and Intelligence',
 'Business Analyst – Regul

In [423]:
df_vacancies["occupational_scope"] = "review"
df_vacancies["scope_reason"] = "manual review required"

In [424]:
core_pattern = (
    r"(?:"
    r"\bdata analyst\b|"
    r"\bbusiness intelligence analyst\b|"
    r"\bbi analyst\b|"
    r"\breporting analyst\b|"
    r"\bdata reporting\b|"
    r"\breporting and insights analyst\b|"
    r"\breporting & insights analyst\b|"
    r"\binsight analyst\b|"
    r"\binsights analyst\b|"
    r"\banalytics analyst\b|"
    r"\bmi analyst\b|"
    r"\bmanagement information analyst\b"
    r")"
)

core_mask = (
    df_vacancies["title_clean"]
    .str.contains(
        core_pattern,
        case=False,
        regex=True,
        na=False
    )
)

df_vacancies.loc[
    core_mask,
    "occupational_scope"
] = "core_analytics"

df_vacancies.loc[
    core_mask,
    "scope_reason"
] = "explicit analytics job title"

In [425]:
adjacent_pattern = (
    r"(?:"
    r"\bbusiness analyst\b|"
    r"\bcommercial analyst\b|"
    r"\bmarketing analyst\b|"
    r"\bpricing analyst\b|"
    r"\bproduct analyst\b|"
    r"\bperformance analyst\b|"
    r"\bcustomer insights?\b|"
    r"\bfinancial reporting analyst\b|"
    r"\bregulatory reporting analyst\b|"
    r"\bdata business analyst\b"
    r")"
)

adjacent_mask = (
    (df_vacancies["occupational_scope"] == "review")
    &
    df_vacancies["title_clean"]
    .str.contains(
        adjacent_pattern,
        case=False,
        regex=True,
        na=False
    )
)

df_vacancies.loc[
    adjacent_mask,
    "occupational_scope"
] = "adjacent_analytics"

df_vacancies.loc[
    adjacent_mask,
    "scope_reason"
] = "analytical role in adjacent occupational family"

In [426]:
df_vacancies["occupational_scope"] == "review"

,occupational_scope
0,False
1,False
2,False
3,False
4,False
...,...
386,True
387,False
388,True
389,True


In [427]:
exclude_pattern = (
    r"(?:"
    r"\bdata engineer\b|"
    r"\bdata engineering\b|"
    r"\bdata scientist\b|"
    r"\bdata science\b|"
    r"\bdata architect\b|"
    r"\bbi developer\b|"
    r"\bsoftware engineer\b|"
    r"\bservice desk\b|"
    r"\basbestos\b|"
    r"\bdemand planner\b|"
    r"\bfinance analyst\b|"
    r"\binvestment analyst\b|"
    r"\bcredit analyst\b"
    r")"
)

exclude_mask = (
    (df_vacancies["occupational_scope"] == "review")
    &
    df_vacancies["title_clean"]
    .str.contains(
        exclude_pattern,
        case=False,
        regex=True,
        na=False
    )
)

df_vacancies.loc[
    exclude_mask,
    "occupational_scope"
] = "exclude_occupation"

df_vacancies.loc[
    exclude_mask,
    "scope_reason"
] = "different occupational family"

In [428]:
df_vacancies["occupational_scope"].value_counts()

,count
occupational_scope,
core_analytics,253
review,84
adjacent_analytics,37
exclude_occupation,17


In [429]:
df_vacancies.groupby(
    "occupational_scope"
)["title"].nunique()

,title
occupational_scope,
adjacent_analytics,25
core_analytics,97
exclude_occupation,10
review,71


In [430]:
review_titles = (
    df_vacancies.loc[
        df_vacancies["occupational_scope"] == "review",
        "title"
    ]
    .value_counts()
)

review_titles.head(80)

,count
title,
Record to Report Ops Analyst - Record To Report,4
Senior Planner,3
Finance Director,2
"Engineering Manager - Data and Analytics, Network Flow and Technologies",2
Analyst,2
...,...
Drainage CCTV Processor,1
Analyst Relations Assistant Manager,1
Membership Director,1


In [431]:
sorted(
    df_vacancies.loc[
        df_vacancies["occupational_scope"] == "review",
        "title"
    ]
    .dropna()
    .unique()
)

['AR Analyst',
 'Account Manager/ Content Producer -Niche B2B Tech Agency',
 'Advanced Specialist, Financial Reporting, Planning and Analysis',
 'Analyst',
 'Analyst O&T Accounting',
 'Analyst Relations Assistant Manager',
 'Analyst – Mandarin / Cantonese',
 'Asset Management Client Reporting Business Change Analyst',
 'Business Intelligence Lead',
 'Business Planning Analyst',
 'Client Tax Reporting Lead Analyst 2',
 'Contract Senior Analyst - Record To Report',
 'Controllers, Non-Financial Regulatory Position Reporting, Senior Analyst, Birmingham',
 'Controllers- Non-Financial Regulatory Position Reporting, Senior Analyst- Birmingham',
 'Customer Success Manager',
 'Cyber Threat Intelligence Analyst | S2 | CISO | Multiple Locations | Internal Mobility',
 'Data & Tracking Analyst (Dunfermline)',
 'Data & Tracking Analyst (Leeds)',
 'Data & Tracking Analyst (Liverpool)',
 'Data and Systems Analyst',
 'Delegate Relations & Sales Executive Investment Week',
 'Drainage CCTV Processor',
 '

In [432]:
core_market = df_vacancies[
    df_vacancies["occupational_scope"]
    == "core_analytics"
]

In [433]:
extended_market = df_vacancies[
    df_vacancies["occupational_scope"]
    .isin([
        "core_analytics",
        "adjacent_analytics"
    ])
]

### 4.4 Manual Resolution of Residual Job Titles

The first-pass occupational classifier intentionally prioritised precision
over recall, leaving ambiguous or uncommon titles for review.

Residual titles were subsequently assigned using explicit title-level rules.
This prevents increasingly broad regular expressions from introducing false
positives and keeps manual classification decisions auditable.

In [434]:
core_title_overrides = [
    "Business Intelligence Lead",
    "Data & Tracking Analyst (Dunfermline)",
    "Data & Tracking Analyst (Leeds)",
    "Data & Tracking Analyst (Liverpool)",
    "Data and Systems Analyst",
    "EMEA Customer & Marketing Insights Lead",
    "Head of Data Strategy and Insights",
    "Lead Analyst - BI/DW - ETL & Reporting Developer",
    "Performance & Reporting Manager",
    "Reporting Solutions Analyst",
    "Senior Analyst Business Intelligence - 3PL Order Management and Finance Operations",
    "Senior Analyst Business Intelligence — 3PL Services Warehouse Operations",
    "UK_Head of Market Access & Business Intelligence"
]

In [435]:
adjacent_title_overrides = [
    "Advanced Specialist, Financial Reporting, Planning and Analysis",
    "Asset Management Client Reporting Business Change Analyst",
    "Business Planning Analyst",
    "Client Tax Reporting Lead Analyst 2",
    "Controllers, Non-Financial Regulatory Position Reporting, Senior Analyst, Birmingham",
    "Controllers- Non-Financial Regulatory Position Reporting, Senior Analyst- Birmingham",
    "ERP and Reporting Specialist",
    "FP&A Analyst",
    "Finance Reporting Intermediate Analyst - C11",
    "Finance Rptg Analyst 2 - C10",
    "Group Catastrophe Portfolio Reporting Senior Analyst",
    "Head of Financial Planning & Analysis",
    "Measurement & Report Analyst-Program and Project Management",
    "OGC Analyst Conflicts Reporting",
    "OGC Analyst – Conflicts Reporting",
    "SIAM Service Reporting and Service Level Analyst",
    "Senior Actuarial Analyst - Investment Reporting",
    "Senior Analyst - CMG - Executive Accelerators – Finance & Reporting",
    "Senior Analyst - Financial Reporting (hybrid)",
    "Senior Analyst - Planning & Reporting",
    "Senior Analyst, External Reporting",
    "Senior Project Controls Reporting & Digital Analyst",
    "Treasury IRRBB Analyst: Reporting Analytics"
]

In [436]:
exclude_title_overrides = [
    "AR Analyst",
    "Account Manager/ Content Producer -Niche B2B Tech Agency",
    "Analyst O&T Accounting",
    "Analyst Relations Assistant Manager",
    "Contract Senior Analyst - Record To Report",
    "Customer Success Manager",
    "Cyber Threat Intelligence Analyst | S2 | CISO | Multiple Locations | Internal Mobility",
    "Delegate Relations & Sales Executive Investment Week",
    "Drainage CCTV Processor",
    "Embedded Cyber Detection and Response Deputy Team Lead",
    "Engineering Manager - Data and Analytics, Network Flow and Technologies",
    "Finance Business Management Sr Analyst - Record To Report",
    "Finance Director",
    "Finance Process & Ops Senior Analyst - Record To Report",
    "Forward Deployed Product Manager, Enterprise",
    "Intelligence Analyst (OSINT)",
    "Loan Analyst",
    "Membership Director",
    "Oracle Technical Support Lead (WMS) - up to £95k  Bonus",
    "Project Manager (eFront)",
    "Record to Report Ops Analyst - Record To Report",
    "Record to Report Ops Analyst-Record To Report",
    "Senior Planner",
    "Senior Salesforce Developer",
    "Software Asset Management (SAM) Analyst",
    "Sr. Manager, B2B Sales Marketing Solutions",
    "Talent Network",
    "Tax & Treasury Analyst",
    "Treasury Analyst"
]

In [437]:
review_mask = (
    df_vacancies["occupational_scope"] == "review"
)

core_override_mask = (
    review_mask
    & df_vacancies["title"].isin(core_title_overrides)
)

df_vacancies.loc[
    core_override_mask,
    "occupational_scope"
] = "core_analytics"

df_vacancies.loc[
    core_override_mask,
    "scope_reason"
] = "manual title review: direct analytics role"

In [438]:
review_mask = (
    df_vacancies["occupational_scope"] == "review"
)

adjacent_override_mask = (
    review_mask
    & df_vacancies["title"].isin(adjacent_title_overrides)
)

df_vacancies.loc[
    adjacent_override_mask,
    "occupational_scope"
] = "adjacent_analytics"

df_vacancies.loc[
    adjacent_override_mask,
    "scope_reason"
] = "manual title review: adjacent analytical role"

In [439]:
review_mask = (
    df_vacancies["occupational_scope"] == "review"
)

exclude_override_mask = (
    review_mask
    & df_vacancies["title"].isin(exclude_title_overrides)
)

df_vacancies.loc[
    exclude_override_mask,
    "occupational_scope"
] = "exclude_occupation"

df_vacancies.loc[
    exclude_override_mask,
    "scope_reason"
] = "manual title review: different occupational family"

In [440]:
len(exclude_title_overrides)

29

In [441]:
print("Core overrides:", len(core_title_overrides))
print("Adjacent overrides:", len(adjacent_title_overrides))
print("Exclude overrides:", len(exclude_title_overrides))

Core overrides: 13
Adjacent overrides: 23
Exclude overrides: 29


In [442]:
df_vacancies["occupational_scope"].value_counts()

,count
occupational_scope,
core_analytics,266
adjacent_analytics,63
exclude_occupation,55
review,7


In [443]:
remaining_review = df_vacancies[
    df_vacancies["occupational_scope"] == "review"
][
    [
        "job_id",
        "title",
        "company",
        "location",
        "description"
    ]
].copy()

print("Records still requiring review:", len(remaining_review))
print(
    "Unique titles still requiring review:",
    remaining_review["title"].nunique()
)

Records still requiring review: 7
Unique titles still requiring review: 6


In [444]:
remaining_review["title"].value_counts()

,count
title,
Analyst,2
Head of Performance Partners,1
Analyst – Mandarin / Cantonese,1
MR__Report Writing_Lead Analyst,1
New Job,1
People operations analyst,1


In [445]:
for _, row in remaining_review.iterrows():

    print("=" * 100)
    print("JOB ID:", row["job_id"])
    print("TITLE:", row["title"])
    print("COMPANY:", row["company"])
    print("LOCATION:", row["location"])
    print()

    print(row["description"][:1500])
    print()

JOB ID: 5812202949
TITLE: Head of Performance Partners
COMPANY: Convergence Group
LOCATION: Blythe Valley Park, Shirley

At Convergence Group, we believe great business performance starts with great people. That's why we've created our Performance POD model - bringing together managers, Performance Partners, Business Intelligence and Business Analysts to make better decisions, develop capability and improve performance across the business. Now we’re looking for a Head of Performance Partners to lead our team and take that model to the next level. This isn't a role where you'll spend your days writing strategy doc…

JOB ID: 5810985595
TITLE: Analyst
COMPANY: GlobalData UK Ltd
LOCATION: London, UK

Financial Services Research and Insights – Data Analyst Who we are… GlobalData operates an intelligence platform that empowers leaders to act decisively in a world of complexity and change. By uniting proprietary data, human expertise, and purpose-built AI into a single, connected platform, we

In [446]:
(df_vacancies["occupational_scope"] == "review").sum()

np.int64(7)

In [447]:
residual_review = (
    df_vacancies[
        df_vacancies["occupational_scope"] == "review"
    ][
        [
            "job_id",
            "title",
            "company",
            "location",
            "salary_min",
            "salary_max",
            "description"
        ]
    ]
    .copy()
)

for _, row in residual_review.iterrows():

    print("=" * 110)
    print(f"JOB ID:   {row['job_id']}")
    print(f"TITLE:    {row['title']}")
    print(f"COMPANY:  {row['company']}")
    print(f"LOCATION: {row['location']}")
    print(
        f"SALARY:   £{row['salary_min']} - "
        f"£{row['salary_max']}"
    )
    print("-" * 110)

    print(row["description"])
    print()

JOB ID:   5812202949
TITLE:    Head of Performance Partners
COMPANY:  Convergence Group
LOCATION: Blythe Valley Park, Shirley
SALARY:   £80000.0 - £95000.0
--------------------------------------------------------------------------------------------------------------
At Convergence Group, we believe great business performance starts with great people. That's why we've created our Performance POD model - bringing together managers, Performance Partners, Business Intelligence and Business Analysts to make better decisions, develop capability and improve performance across the business. Now we’re looking for a Head of Performance Partners to lead our team and take that model to the next level. This isn't a role where you'll spend your days writing strategy doc…

JOB ID:   5810985595
TITLE:    Analyst
COMPANY:  GlobalData UK Ltd
LOCATION: London, UK
SALARY:   £43550.27 - £43550.27
--------------------------------------------------------------------------------------------------------------


In [448]:
final_manual_scope = {
    5812202949: (
        "adjacent_analytics",
        "performance management role supported by BI and analytics"
    ),

    5810985595: (
        "core_analytics",
        "description explicitly identifies the role as Data Analyst"
    ),

    5812202801: (
        "core_analytics",
        "description explicitly identifies the role as Data Analyst"
    ),

    5788127876: (
        "exclude_occupation",
        "investigative due diligence and qualitative research, not data analytics"
    ),

    5210489136: (
        "adjacent_analytics",
        "reporting-focused analytical role; core data-analysis duties not explicit in snippet"
    ),

    5838262200: (
        "core_analytics",
        "description explicitly describes Data Analyst work using Power BI and reporting"
    ),

    5813152277: (
        "adjacent_analytics",
        "people operations role with substantial reporting and people analytics responsibilities"
    )
}

for job_id, (scope, reason) in final_manual_scope.items():

    mask = df_vacancies["job_id"] == job_id

    df_vacancies.loc[
        mask,
        "occupational_scope"
    ] = scope

    df_vacancies.loc[
        mask,
        "scope_reason"
    ] = reason

In [449]:
df_vacancies["occupational_scope"].value_counts()

,count
occupational_scope,
core_analytics,269
adjacent_analytics,66
exclude_occupation,56


In [450]:
print(
    "Records still requiring review:",
    (df_vacancies["occupational_scope"] == "review").sum()
)

Records still requiring review: 0


In [451]:
df_vacancies[
    df_vacancies["job_id"].isin(
        final_manual_scope.keys()
    )
][
    [
        "job_id",
        "title",
        "company",
        "occupational_scope",
        "scope_reason"
    ]
]

,job_id,title,company,occupational_scope,scope_reason
190,5812202949,Head of Performance Partners,Convergence Group,adjacent_analytics,performance management role supported by BI and analytics
198,5810985595,Analyst,GlobalData UK Ltd,core_analytics,description explicitly identifies the role as Data Analyst
199,5812202801,Analyst,GlobalData PLC,core_analytics,description explicitly identifies the role as Data Analyst
202,5788127876,Analyst – Mandarin / Cantonese,Anthesis Group,exclude_occupation,"investigative due diligence and qualitative research, not data analytics"
294,5210489136,MR__Report Writing_Lead Analyst,dentsu,adjacent_analytics,reporting-focused analytical role; core data-analysis duties not explicit in snippet
336,5838262200,New Job,Red Sky Personnel Ltd,core_analytics,description explicitly describes Data Analyst work using Power BI and reporting
372,5813152277,People operations analyst,Annapurna HR Ltd,adjacent_analytics,people operations role with substantial reporting and people analytics responsibilities


In [452]:
globaldata_check = df_vacancies[
    df_vacancies["job_id"].isin(
        [5810985595, 5812202801]
    )
][
    [
        "job_id",
        "title",
        "company",
        "location",
        "created",
        "salary_min",
        "salary_max",
        "salary_is_predicted",
        "description_norm"
    ]
].copy()

globaldata_check

,job_id,title,company,location,created,salary_min,salary_max,salary_is_predicted,description_norm
198,5810985595,Analyst,GlobalData UK Ltd,"London, UK",2026-07-22 16:28:30+00:00,43550.27,43550.27,1,financial services research and insights data analyst who we are globaldata operates an intelligence platform that empowers leaders to act decisively in a world of complexity and change by uniting proprietary data human expertise and purpose built ai into a single connected platform we help organizations see what s coming move faster and lead with confidence our solutions are used by over 5 000 organizations across the world s largest industries delivering tailored intelligence that
199,5812202801,Analyst,GlobalData PLC,"Fleet Street, Central London",2026-07-23 11:31:10+00:00,54764.32,54764.32,1,financial services research and insights data analyst who we are globaldata operates an intelligence platform that empowers leaders to act decisively in a world of complexity and change by uniting proprietary data human expertise and purpose built ai into a single connected platform we help organizations see what s coming move faster and lead with confidence our solutions are used by over 5 000 organizations across the world s largest industries delivering tailored intelligence that


In [453]:
print(
    "Unique normalized descriptions:",
    globaldata_check["description_norm"].nunique()
)

print(
    "Time difference:",
    globaldata_check["created"].max()
    - globaldata_check["created"].min()
)

Unique normalized descriptions: 1
Time difference: 0 days 19:02:40


In [454]:
core_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        == "core_analytics"
    ]
    .copy()
)

extended_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        .isin([
            "core_analytics",
            "adjacent_analytics"
        ])
    ]
    .copy()
)

In [455]:
globaldata_duplicate_audit = globaldata_check.copy()

globaldata_duplicate_audit["duplicate_resolution"] = (
    "same vacancy - identical description, "
    "same London market, posted within 24 hours"
)

globaldata_duplicate_audit["keep_record"] = (
    globaldata_duplicate_audit["created"]
    == globaldata_duplicate_audit["created"].max()
)

globaldata_duplicate_audit[
    [
        "job_id",
        "company",
        "location",
        "created",
        "salary_min",
        "salary_max",
        "keep_record"
    ]
]

,job_id,company,location,created,salary_min,salary_max,keep_record
198,5810985595,GlobalData UK Ltd,"London, UK",2026-07-22 16:28:30+00:00,43550.27,43550.27,False
199,5812202801,GlobalData PLC,"Fleet Street, Central London",2026-07-23 11:31:10+00:00,54764.32,54764.32,True


In [456]:
late_duplicate_job_ids = [5810985595]

df_vacancies = (
    df_vacancies[
        ~df_vacancies["job_id"].isin(late_duplicate_job_ids)
    ]
    .reset_index(drop=True)
    .copy()
)

In [457]:
print("Final vacancy records:", len(df_vacancies))
print("Unique job IDs:", df_vacancies["job_id"].nunique())
print(
    "Records still requiring review:",
    (df_vacancies["occupational_scope"] == "review").sum()
)

Final vacancy records: 390
Unique job IDs: 390
Records still requiring review: 0


### 4.5 Late-Stage Duplicate Resolution

During occupational review, two GlobalData advertisements were found to have
identical normalized descriptions and to represent the same Data Analyst role
within the London market.

The records were posted approximately 19 hours apart under slightly different
company and location labels. Both salary values were Adzuna-predicted rather
than explicitly advertised.

The later advertisement was therefore retained as the canonical record.

This additional case demonstrates a limitation of exact location-based
duplicate blocking: the same vacancy may be represented using different
geographic granularity or employer naming conventions.

In [458]:
df_vacancies["occupational_scope"].value_counts()

,count
occupational_scope,
core_analytics,268
adjacent_analytics,66
exclude_occupation,56


In [459]:
core_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        == "core_analytics"
    ]
    .reset_index(drop=True)
    .copy()
)

extended_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        .isin([
            "core_analytics",
            "adjacent_analytics"
        ])
    ]
    .reset_index(drop=True)
    .copy()
)

In [460]:
print("All vacancy records:", len(df_vacancies))
print("Core analytics market:", len(core_market))
print("Extended analytics market:", len(extended_market))
print(
    "Excluded occupations:",
    (df_vacancies["occupational_scope"] == "exclude_occupation").sum()
)

All vacancy records: 390
Core analytics market: 268
Extended analytics market: 334
Excluded occupations: 56


In [461]:
assert len(df_vacancies) == df_vacancies["job_id"].nunique()
assert (df_vacancies["occupational_scope"] == "review").sum() == 0

print("Final occupational dataset validation passed.")

Final occupational dataset validation passed.


## 5. Job Family Standardisation

After occupational relevance was resolved, relevant vacancies were grouped
into analytical job families.

The original advertised title is preserved. Standardised job families are
derived only for analytical comparison and do not replace the source title.

Job family and seniority are treated as separate dimensions because titles
such as "Senior Data Analyst" and "Junior Data Analyst" belong to the same
occupational family but represent different career levels.

In [462]:
core_market = (
    df_vacancies[
        df_vacancies["occupational_scope"] == "core_analytics"
    ]
    .reset_index(drop=True)
    .copy()
)

extended_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        .isin([
            "core_analytics",
            "adjacent_analytics"
        ])
    ]
    .reset_index(drop=True)
    .copy()
)

print("Core market:", len(core_market))
print("Extended market:", len(extended_market))

Core market: 268
Extended market: 334


In [463]:
core_market["title"].value_counts().head(60)

,count
title,
Data Analyst,108
Business Intelligence Analyst,12
Financial Reporting Analyst,8
Lead Data Analyst/Project Controller,4
Regulatory Reporting Analyst,4
Data & Reporting Analyst,3
BI Analyst,3
Senior Data Analyst,3
Senior Reporting Analyst,3


In [464]:
print(
    "Unique core-market titles:",
    core_market["title"].nunique()
)

Unique core-market titles: 112


In [465]:
def classify_job_family(title):

    if pd.isna(title):
        return "Other Analytics"

    title = normalize_text(title)

    # Business Intelligence
    if (
        "business intelligence" in title
        or re.search(r"\bbi analyst\b", title)
        or re.search(r"\bbi data analyst\b", title)
        or re.search(r"\bbi reporting analyst\b", title)
    ):
        return "Business Intelligence"

    # Insights
    if (
        "insight analyst" in title
        or "insights analyst" in title
        or "customer insights" in title
        or "marketing insights" in title
        or "data strategy and insights" in title
    ):
        return "Insights Analytics"

    # Reporting
    if (
        "reporting analyst" in title
        or "data reporting" in title
        or "reporting solutions analyst" in title
        or "report writing" in title
    ):
        return "Reporting Analytics"

    # MI / Management Information
    if (
        re.search(r"\bmi analyst\b", title)
        or "management information analyst" in title
    ):
        return "Management Information"

    # Data Analyst
    if "data analyst" in title:
        return "Data Analyst"

    return "Other Analytics"

In [466]:
core_market["job_family"] = (
    core_market["title"]
    .apply(classify_job_family)
)

In [467]:
core_market["job_family"].value_counts()

,count
job_family,
Data Analyst,137
Reporting Analytics,69
Business Intelligence,45
Other Analytics,9
Insights Analytics,7
Management Information,1


In [468]:
other_analytics_titles = (
    core_market.loc[
        core_market["job_family"] == "Other Analytics",
        "title"
    ]
    .value_counts()
)

other_analytics_titles

,count
title,
Data and Systems Analyst,1
Analyst,1
Lead Analyst - BI/DW - ETL & Reporting Developer,1
"HR MI, Systems & Analytics Analyst",1
New Job,1
Performance & Reporting Manager,1
Data & Tracking Analyst (Dunfermline),1
Data & Tracking Analyst (Leeds),1
Data & Tracking Analyst (Liverpool),1


In [469]:
def classify_seniority(title):

    if pd.isna(title):
        return "Unspecified"

    title = normalize_text(title)

    if re.search(
        r"\b(head|director|chief)\b",
        title
    ):
        return "Head / Director"

    if re.search(
        r"\b(manager|managerial)\b",
        title
    ):
        return "Manager"

    if re.search(
        r"\b(principal|lead)\b",
        title
    ):
        return "Lead / Principal"

    if re.search(
        r"\bsenior\b|\bsr\b",
        title
    ):
        return "Senior"

    if re.search(
        r"\b(junior|jr)\b",
        title
    ):
        return "Junior"

    if re.search(
        r"\b(trainee|graduate|intern|internship|placement)\b",
        title
    ):
        return "Entry / Trainee"

    return "Unspecified"

In [470]:
core_market["seniority"] = (
    core_market["title"]
    .apply(classify_seniority)
)

In [471]:
core_market["seniority"].value_counts()

,count
seniority,
Unspecified,224
Senior,26
Lead / Principal,10
Junior,5
Entry / Trainee,1
Manager,1
Head / Director,1


In [472]:
pd.crosstab(
    core_market["job_family"],
    core_market["seniority"]
)

seniority,Entry / Trainee,Head / Director,Junior,Lead / Principal,Manager,Senior,Unspecified
job_family,,,,,,,
Business Intelligence,0,0,0,3,0,9,33
Data Analyst,1,0,2,4,0,6,124
Insights Analytics,0,1,0,1,0,1,4
Management Information,0,0,0,0,0,0,1
Other Analytics,0,0,0,1,1,0,7
Reporting Analytics,0,0,3,1,0,10,55


In [473]:
print("Core market:", len(core_market))
print("Unique core titles:", core_market["title"].nunique())

Core market: 268
Unique core titles: 112


In [474]:
core_market["job_family"].value_counts()

,count
job_family,
Data Analyst,137
Reporting Analytics,69
Business Intelligence,45
Other Analytics,9
Insights Analytics,7
Management Information,1


In [475]:
other_analytics_titles

,count
title,
Data and Systems Analyst,1
Analyst,1
Lead Analyst - BI/DW - ETL & Reporting Developer,1
"HR MI, Systems & Analytics Analyst",1
New Job,1
Performance & Reporting Manager,1
Data & Tracking Analyst (Dunfermline),1
Data & Tracking Analyst (Leeds),1
Data & Tracking Analyst (Liverpool),1


In [476]:
core_market["seniority"].value_counts()

,count
seniority,
Unspecified,224
Senior,26
Lead / Principal,10
Junior,5
Entry / Trainee,1
Manager,1
Head / Director,1


### 4.6 Occupational Scope Precedence Correction

A review of the final Core Analytics titles identified a classification
precedence issue.

The broad "reporting analyst" rule had classified some finance, regulatory,
risk and specialist reporting occupations as Core Analytics before the
adjacent-occupation rules were evaluated.

These domain-specific reporting roles were reassigned to Adjacent Analytics.
This preserves analytical relevance while preventing specialist finance and
regulatory occupations from inflating the core Data Analytics market.

In [477]:
adjacent_reporting_corrections = [
    "Financial Reporting Analyst",
    "Regulatory Reporting Analyst",
    "Senior Financial Planning and Reporting Analyst",
    "Regulatory Reporting Analyst - Insurance",
    "Regulatory Reporting Analyst (12 month FTC)",
    "Finance Analyst - Data & Reporting",
    "Risk & Reporting Analyst",
    "Group CAT Portfolio reporting Analyst - Selby Jennings",
    "Client Reporting Analyst - Mason Blake"
]

In [478]:
correction_mask = (
    df_vacancies["title"]
    .isin(adjacent_reporting_corrections)
)

df_vacancies.loc[
    correction_mask,
    "title"
].value_counts()

,count
title,
Financial Reporting Analyst,8
Regulatory Reporting Analyst,4
Risk & Reporting Analyst,2
Senior Financial Planning and Reporting Analyst,2
Regulatory Reporting Analyst - Insurance,2
Regulatory Reporting Analyst (12 month FTC),2
Finance Analyst - Data & Reporting,2
Client Reporting Analyst - Mason Blake,1
Group CAT Portfolio reporting Analyst - Selby Jennings,1


In [479]:
df_vacancies.loc[
    correction_mask,
    "occupational_scope"
] = "adjacent_analytics"

df_vacancies.loc[
    correction_mask,
    "scope_reason"
] = (
    "specialist finance, regulatory, risk or "
    "client-reporting occupational family"
)

In [480]:
df_vacancies["occupational_scope"].value_counts()

,count
occupational_scope,
core_analytics,244
adjacent_analytics,90
exclude_occupation,56


In [481]:
core_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        == "core_analytics"
    ]
    .reset_index(drop=True)
    .copy()
)

extended_market = (
    df_vacancies[
        df_vacancies["occupational_scope"]
        .isin([
            "core_analytics",
            "adjacent_analytics"
        ])
    ]
    .reset_index(drop=True)
    .copy()
)

print("Core market:", len(core_market))
print("Extended market:", len(extended_market))

Core market: 244
Extended market: 334


In [482]:
core_market["job_family"] = (
    core_market["title"]
    .apply(classify_job_family)
)

In [483]:
core_market["seniority"] = (
    core_market["title"]
    .apply(classify_seniority)
)

In [484]:
core_market["job_family"].value_counts()

,count
job_family,
Data Analyst,137
Business Intelligence,45
Reporting Analytics,45
Other Analytics,9
Insights Analytics,7
Management Information,1


In [485]:
family_title_overrides = {
    "Data and Systems Analyst": "Data Analyst",

    "Lead Analyst - BI/DW - ETL & Reporting Developer":
        "Business Intelligence",

    "HR MI, Systems & Analytics Analyst":
        "Management Information",

    "Performance & Reporting Manager":
        "Reporting Analytics",

    "Data & Tracking Analyst (Dunfermline)":
        "Data Analyst",

    "Data & Tracking Analyst (Leeds)":
        "Data Analyst",

    "Data & Tracking Analyst (Liverpool)":
        "Data Analyst"
}

In [486]:
for title, family in family_title_overrides.items():

    core_market.loc[
        core_market["title"] == title,
        "job_family"
    ] = family

In [487]:
family_job_id_overrides = {
    5812202801: "Data Analyst",
    5838262200: "Data Analyst"
}

for job_id, family in family_job_id_overrides.items():

    core_market.loc[
        core_market["job_id"] == job_id,
        "job_family"
    ] = family

In [488]:
core_market["job_family"].value_counts()

,count
job_family,
Data Analyst,143
Business Intelligence,46
Reporting Analytics,46
Insights Analytics,7
Management Information,2


In [489]:
core_market.loc[
    core_market["job_family"] == "Other Analytics",
    ["job_id", "title", "company"]
]

,job_id,title,company


In [490]:
print("Core market:", len(core_market))
print("Extended market:", len(extended_market))

print()
print(core_market["job_family"].value_counts())

print()
print(core_market["seniority"].value_counts())

print()
print(
    "Other Analytics remaining:",
    (core_market["job_family"] == "Other Analytics").sum()
)

Core market: 244
Extended market: 334

job_family
Data Analyst              143
Business Intelligence      46
Reporting Analytics        46
Insights Analytics          7
Management Information      2
Name: count, dtype: int64

seniority
Unspecified         202
Senior               24
Lead / Principal     10
Junior                5
Entry / Trainee       1
Manager               1
Head / Director       1
Name: count, dtype: int64

Other Analytics remaining: 0


## 6. Final Data Quality Validation

Before exporting the processed datasets, final validation checks were
performed to confirm record uniqueness, occupational classification
completeness and consistency of the derived analytical dimensions.

No seniority level was inferred when it was not explicitly indicated by the
advertised job title.

In [491]:
print("Final classified vacancies:", len(df_vacancies))
print("Unique vacancy job IDs:", df_vacancies["job_id"].nunique())

print()
print("Core analytics:", len(core_market))
print("Extended analytics:", len(extended_market))

print()
print(
    "Unresolved occupational records:",
    (df_vacancies["occupational_scope"] == "review").sum()
)

print(
    "Unclassified core job families:",
    core_market["job_family"].isna().sum()
)

print(
    "Other Analytics remaining:",
    (core_market["job_family"] == "Other Analytics").sum()
)

print(
    "Missing seniority classifications:",
    core_market["seniority"].isna().sum()
)

Final classified vacancies: 390
Unique vacancy job IDs: 390

Core analytics: 244
Extended analytics: 334

Unresolved occupational records: 0
Unclassified core job families: 0
Other Analytics remaining: 0
Missing seniority classifications: 0


In [492]:
core_market["salary_midpoint"] = (
    core_market[["salary_min", "salary_max"]]
    .mean(axis=1)
)

In [493]:
extended_market["salary_midpoint"] = (
    extended_market[["salary_min", "salary_max"]]
    .mean(axis=1)
)

In [494]:
core_market[
    [
        "salary_min",
        "salary_max",
        "salary_midpoint",
        "salary_source"
    ]
].head(10)

,salary_min,salary_max,salary_midpoint,salary_source
0,53889.32,53889.32,53889.32,Adzuna predicted
1,53416.18,53416.18,53416.18,Adzuna predicted
2,63011.66,63011.66,63011.66,Adzuna predicted
3,40000.00,40000.00,40000.00,Advertised / non-predicted
4,30000.00,30000.00,30000.00,Advertised / non-predicted
5,26645.01,26645.01,26645.01,Adzuna predicted
6,40000.00,40000.00,40000.00,Advertised / non-predicted
7,30000.00,30000.00,30000.00,Advertised / non-predicted
8,56400.93,56400.93,56400.93,Adzuna predicted
9,46500.00,46500.00,46500.00,Advertised / non-predicted


In [495]:
core_market["has_salary"] = (
    core_market["salary_min"].notna()
    | core_market["salary_max"].notna()
)

core_market["advertised_salary"] = (
    core_market["salary_is_predicted"] == 0
)

In [496]:
print(
    "Core jobs with salary data:",
    core_market["has_salary"].sum()
)

print(
    "Advertised / non-predicted salary:",
    core_market["advertised_salary"].sum()
)

print(
    "Adzuna-predicted salary:",
    (core_market["salary_is_predicted"] == 1).sum()
)

Core jobs with salary data: 244
Advertised / non-predicted salary: 120
Adzuna-predicted salary: 124


In [497]:
pipeline_summary = pd.DataFrame({
    "stage": [
        "Raw API observations",
        "Unique Adzuna job IDs",
        "After cross-ID deduplication",
        "After late duplicate resolution",
        "Confirmed training/programme ads",
        "Final classified vacancies",
        "Extended analytics market",
        "Core analytics market"
    ],
    "records": [
        532,
        491,
        487,
        486,
        96,
        390,
        len(extended_market),
        len(core_market)
    ]
})

pipeline_summary

,stage,records
0,Raw API observations,532
1,Unique Adzuna job IDs,491
2,After cross-ID deduplication,487
3,After late duplicate resolution,486
4,Confirmed training/programme ads,96
5,Final classified vacancies,390
6,Extended analytics market,334
7,Core analytics market,244


In [498]:
processed_path = f"{PROJECT_PATH}/data/processed"

In [499]:
df_vacancies.to_csv(
    f"{processed_path}/uk_jobs_classified.csv",
    index=False
)

In [500]:
core_market.to_csv(
    f"{processed_path}/uk_core_analytics_jobs.csv",
    index=False
)

In [501]:
extended_market.to_csv(
    f"{processed_path}/uk_extended_analytics_jobs.csv",
    index=False
)

In [502]:
job_search_terms.to_csv(
    f"{processed_path}/job_search_terms.csv",
    index=False
)

In [503]:
import os

files_to_check = [
    "uk_jobs_classified.csv",
    "uk_core_analytics_jobs.csv",
    "uk_extended_analytics_jobs.csv",
    "job_search_terms.csv"
]

for filename in files_to_check:
    path = f"{processed_path}/{filename}"

    print(
        filename,
        "->",
        os.path.exists(path)
    )

uk_jobs_classified.csv -> True
uk_core_analytics_jobs.csv -> True
uk_extended_analytics_jobs.csv -> True
job_search_terms.csv -> True
